In [ ]:
import pandas as pd
import os
from functools import reduce
from io import BytesIO

import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# UPLOAD DO ARQUIVO
upload_widget = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV')
use_all_toggle = widgets.ToggleButtons(
    options=['Todos os dados', 'Definir intervalo'],
    description='Intervalo:',
    style={'description_width': 'initial'}
)
year_start_widget = widgets.BoundedIntText(value=2000, min=1800, max=2100, description='Ano inicial:')
year_end_widget   = widgets.BoundedIntText(value=2024, min=1800, max=2100, description='Ano final:')
year_range_box    = widgets.VBox([year_start_widget, year_end_widget])

threshold_widget = widgets.BoundedIntText(
    value=25, min=1, max=31,
    description='Dias mínimos válidos por mês:',
    style={'description_width': 'initial'}
)

run_button      = widgets.Button(description='Processar', button_style='primary', icon='play')
download_button = widgets.Button(description='Baixar Excel', button_style='success', icon='download', disabled=True)
output_area     = widgets.Output()

# Mostra/oculta seletor de intervalo conforme escolha
def on_toggle_change(change):
    year_range_box.layout.display = 'none' if change['new'] == 'Todos os dados' else 'block'

use_all_toggle.observe(on_toggle_change, names='value')
year_range_box.layout.display = 'none'

# Reseta o estado ao carregar novo arquivo
def on_new_upload(change):
    if change['new']:
        output_buffer.clear()
        download_button.disabled = True
        with output_area:
            clear_output()

upload_widget.observe(on_new_upload, names='value')

display(widgets.VBox([
    upload_widget,
    use_all_toggle,
    year_range_box,
    threshold_widget,
    run_button,
    download_button,
    output_area
]))

# Variável para guardar o arquivo gerado em memória
output_buffer = {}

# FUNÇÕES: CÁLCULOS COM SUFIXOS DE COLUNAS
def calc_monthly_by_year(df_subset, suffix):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Ano', 'Mes'])

    result = df_subset[['EstacaoCodigo', 'Ano', 'Mes', 'Total Mensal', 'Média Diária',
                         'Desvio Padrão Diário', 'Mediana Diária', 'Mínimo Diário',
                         'Máximo Diário', 'Dias com Chuva', 'Registros Válidos']].copy()
    result = result.rename(columns={
        'Total Mensal':         f'Total Mensal{suffix}',
        'Média Diária':         f'Média Diária{suffix}',
        'Desvio Padrão Diário': f'Desvio Padrão Diário{suffix}',
        'Mediana Diária':       f'Mediana Diária{suffix}',
        'Mínimo Diário':        f'Mínimo Diário{suffix}',
        'Máximo Diário':        f'Máximo Diário{suffix}',
        'Dias com Chuva':       f'Dias com Chuva{suffix}',
        'Registros Válidos':    f'Registros Válidos{suffix}'
    })
    return result

def calc_historical_monthly(df_subset, suffix, threshold):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Mes'])

    # Exclui meses com registros insuficientes
    df_valid = df_subset[df_subset['Registros Válidos'] >= threshold]

    if df_valid.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Mes'])

    result = df_valid.groupby(['EstacaoCodigo', 'Mes']).agg(**{
        f'Média Histórica{suffix}':         ('Total Mensal', 'mean'),
        f'Desvio Padrão Histórico{suffix}': ('Total Mensal', 'std'),
        f'Mediana Histórica{suffix}':       ('Total Mensal', 'median'),
        f'Mínimo Histórico{suffix}':        ('Total Mensal', 'min'),
        f'Máximo Histórico{suffix}':        ('Total Mensal', 'max'),
        f'Anos Válidos{suffix}':            ('Total Mensal', 'count'),
        f'Total Registros Válidos{suffix}': ('Registros Válidos', 'sum'),
        f'Total Dias com Chuva{suffix}':    ('Dias com Chuva', 'sum')
    }).reset_index()
    return result

def calc_historical_annual(df_subset, suffix, threshold):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Ano'])

    # Exclui meses com registros insuficientes antes de agregar por ano
    df_valid = df_subset[df_subset['Registros Válidos'] >= threshold]

    # Conta quantos meses válidos cada ano possui após o filtro
    valid_months_per_year = (
        df_valid.groupby(['EstacaoCodigo', 'Ano'])['Mes']
        .count()
        .reset_index()
        .rename(columns={'Mes': 'Meses Válidos'})
    )

    result = df_valid.groupby(['EstacaoCodigo', 'Ano']).agg(**{
        f'Total Acumulado{suffix}':         ('Total Mensal', 'sum'),
        f'Média Mensal{suffix}':            ('Total Mensal', 'mean'),
        f'Desvio Padrão Mensal{suffix}':    ('Total Mensal', 'std'),
        f'Mediana Mensal{suffix}':          ('Total Mensal', 'median'),
        f'Mínimo Mensal{suffix}':           ('Total Mensal', 'min'),
        f'Máximo Mensal{suffix}':           ('Total Mensal', 'max'),
        f'Total Registros Válidos{suffix}': ('Registros Válidos', 'sum'),
        f'Total Dias com Chuva{suffix}':    ('Dias com Chuva', 'sum')
    }).reset_index()

    # Adiciona coluna de meses válidos para transparência
    result = result.merge(valid_months_per_year, on=['EstacaoCodigo', 'Ano'], how='left')
    result = result.rename(columns={'Meses Válidos': f'Meses Válidos{suffix}'})
    return result

# LÓGICA PRINCIPAL
def on_run_clicked(b):
    output_buffer.clear()
    download_button.disabled = True

    with output_area:
        clear_output()

        # Verifica se arquivo foi carregado
        if not upload_widget.value:
            print("Nenhum arquivo carregado. Por favor, faça o upload de um CSV.")
            return

        if isinstance(upload_widget.value, dict):
            input_filename = list(upload_widget.value.keys())[0]
            uploaded_file  = list(upload_widget.value.values())[0]
            raw_bytes      = uploaded_file['content']
        else:
            uploaded_file  = upload_widget.value[0]
            input_filename = uploaded_file['name']
            raw_bytes      = bytes(uploaded_file['content'])

        # Lê o CSV considerando vírgulas como decimais
        df = pd.read_csv(BytesIO(raw_bytes), encoding='latin1', sep=';', skiprows=14, decimal=',')

        # EXCLUIR COLUNAS DE METADADOS
        rain_cols = [f'Chuva{i:02d}' for i in range(1, 32)]
        essential_cols = ['EstacaoCodigo', 'Data', 'NivelConsistencia']

        available_cols = [c for c in essential_cols + rain_cols if c in df.columns]
        df = df[available_cols].copy()

        date_column = 'Data'

        if date_column not in df.columns:
            print(f"AVISO: Coluna '{date_column}' não encontrada! Verifique o arquivo original.")
            return

        # Extrai Ano e Mês
        df['Data_dt'] = pd.to_datetime(df[date_column], format='%d/%m/%Y', errors='coerce')
        df['Ano'] = df['Data_dt'].dt.year
        df['Mes'] = df['Data_dt'].dt.month
        df = df.dropna(subset=['Ano', 'Mes'])

        # Converte Mês e Ano para inteiros (sem casas decimais)
        df['Ano'] = df['Ano'].astype(int)
        df['Mes'] = df['Mes'].astype(int)

        # Apaga a coluna original de data
        df = df.drop(columns=[date_column, 'Data_dt'])

        # SELEÇÃO DO INTERVALO DE ANÁLISE
        year_min = int(df['Ano'].min())
        year_max = int(df['Ano'].max())
        print(f"Dados disponíveis de {year_min} a {year_max}.")

        use_all = use_all_toggle.value == 'Todos os dados'
        if use_all:
            year_start, year_end = year_min, year_max
            print(f"Usando todos os dados ({year_min}–{year_max}, {len(df)} registros).")
        else:
            year_start = year_start_widget.value
            year_end   = year_end_widget.value

            if not (year_min <= year_start <= year_end <= year_max):
                print(f"Intervalo inválido. Use valores entre {year_min} e {year_max}, com início ≤ fim.")
                return

            df = df[(df['Ano'] >= year_start) & (df['Ano'] <= year_end)]
            print(f"Análise restrita a {year_start}–{year_end} ({len(df)} registros).")

        # LIMIAR DE QUALIDADE DE DADOS
        threshold = threshold_widget.value

        # Calcula as estatísticas diárias
        rain_cols = [c for c in df.columns if (c.startswith('Chuva') and not c.endswith('Status'))]
        df['Total Mensal']         = df[rain_cols].sum(axis=1, skipna=True)
        df['Média Diária']         = df[rain_cols].mean(axis=1, skipna=True)
        df['Desvio Padrão Diário'] = df[rain_cols].std(axis=1, skipna=True)
        df['Mediana Diária']       = df[rain_cols].median(axis=1)
        df['Mínimo Diário']        = df[rain_cols].min(axis=1)
        df['Máximo Diário']        = df[rain_cols].max(axis=1)
        df['Dias com Chuva']       = df[rain_cols].apply(lambda row: (row > 0).sum(), axis=1)
        df['Registros Válidos']    = df[rain_cols].notna().sum(axis=1)

        # SEPARAÇÃO DOS DADOS E APLICAÇÃO DOS CÁLCULOS
        df_raw        = df[df['NivelConsistencia'] == 1].drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])
        df_consistent = df[df['NivelConsistencia'] == 2].drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])
        df_all        = df.sort_values(['NivelConsistencia'], ascending=False).drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])

        # --- TABELA 2: MENSAL POR ANO ---
        dfs_ma = [d for d in [
            calc_monthly_by_year(df_all,        ''),
            calc_monthly_by_year(df_raw,        ' Bruto'),
            calc_monthly_by_year(df_consistent, ' Consistido')
        ] if not d.empty]

        monthly_by_year = (
            reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Ano', 'Mes'], how='outer'), dfs_ma)
            .sort_values(['Ano', 'Mes'], ascending=[False, False])
        ) if dfs_ma else pd.DataFrame()

        # --- TABELA 3: HISTÓRICO MENSAL ---
        dfs_hm = [d for d in [
            calc_historical_monthly(df_all,        '',            threshold),
            calc_historical_monthly(df_raw,        ' Bruto',      threshold),
            calc_historical_monthly(df_consistent, ' Consistido', threshold)
        ] if not d.empty]

        monthly_stats = (
            reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Mes'], how='outer'), dfs_hm)
            .sort_values(['Mes'], ascending=True)
        ) if dfs_hm else pd.DataFrame()

        # --- TABELA 4: HISTÓRICO ANUAL ---
        dfs_ha = [d for d in [
            calc_historical_annual(df_all,        '',            threshold),
            calc_historical_annual(df_raw,        ' Bruto',      threshold),
            calc_historical_annual(df_consistent, ' Consistido', threshold)
        ] if not d.empty]

        annual_stats = (
            reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Ano'], how='outer'), dfs_ha)
            .sort_values(['Ano'], ascending=False)
        ) if dfs_ha else pd.DataFrame()

        # REORGANIZAÇÃO DA ABA PRINCIPAL (Dados Completos)

        # Ordena os dados originais do mais recente para o mais antigo
        df = df.sort_values(['Ano', 'Mes', 'NivelConsistencia'], ascending=[False, False, False])

        # Define a nova ordem exata das colunas
        base_cols       = ['EstacaoCodigo', 'Mes', 'Ano', 'NivelConsistencia']
        calculated_cols = ['Total Mensal'] # Apenas o Total foi mantido
        new_order       = [col for col in base_cols + calculated_cols + rain_cols if col in df.columns]

        # Aplica a nova ordem e remove Média e Desvio Padrão do DataFrame principal
        df = df[new_order]

        # EXPORTAR PARA EXCEL (.xlsx) EM MEMÓRIA
        time_range      = f"{year_start}_{year_end}"
        base_name, _    = os.path.splitext(input_filename)
        output_filename = f"Calculos_{base_name}_{time_range}.xlsx"

        excel_buffer = BytesIO()
        with pd.ExcelWriter(excel_buffer, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Dados_Completos', index=False)
            if not monthly_by_year.empty:
                monthly_by_year.to_excel(writer, sheet_name='Mensal_Por_Ano',   index=False)
                monthly_stats.to_excel(  writer, sheet_name='Historico_Mensal', index=False)
                annual_stats.to_excel(   writer, sheet_name='Historico_Anual',  index=False)

        excel_buffer.seek(0)
        output_buffer['data']     = excel_buffer.read()
        output_buffer['filename'] = output_filename

        print(f"Arquivo pronto: {output_filename}")
        print("Os dados estão ordenados por data decrescente nas abas Dados Completos, Mensal_Por_Ano e Historico_Anual.")
        download_button.disabled = False

# LÓGICA DO BOTÃO DE DOWNLOAD
def on_download_clicked(b):
    with output_area:
        if not output_buffer:
            print("Nenhum arquivo gerado ainda.")
            return

        # Codifica o arquivo em base64 e oferece download via link HTML
        import base64
        from IPython.display import HTML

        b64   = base64.b64encode(output_buffer['data']).decode()
        fname = output_buffer['filename']
        mime  = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'

        display(HTML(f'''
            <a download="{fname}" href="data:{mime};base64,{b64}"
               style="display:inline-block;padding:8px 16px;background:#28a745;
                      color:white;border-radius:4px;text-decoration:none;font-weight:bold;">
                Clique aqui para baixar {fname}
            </a>
        '''))

run_button.on_click(on_run_clicked)
download_button.on_click(on_download_clicked)